# Import Modules

In [1]:
import importlib
import os
import sys

import joblib
import numpy as np
import pandas as pd
import polars as pl

In [2]:
os.chdir("../")
sys.path.insert(0, os.getcwd())

In [7]:
from morai.forecast import constraint
from morai.utils import custom_logger, helpers

In [8]:
logger = custom_logger.setup_logging(__name__)

In [9]:
# update log level if wanting more logging
custom_logger.set_log_level("INFO")

In [10]:
pd.options.display.float_format = "{:,.2f}".format

# Table Rates

In [11]:
rate_filename = helpers.FILES_PATH / "rates" / "rate_map.yaml"

In [25]:
mt = tables.MortTable(rate="glm_mults", rate_filename=rate_filename)

 2025-06-05 23:44:39 | morai.experience.tables | INFO     | loading 'glm_mults' from mapping file: C:\Users\johnk\Desktop\github\morai\files\rates\rate_map.yaml 
 2025-06-05 23:44:39 | morai.experience.tables | INFO     | building table for rate: 'glm_mults' with format: 'workbook' 


In [58]:
rate_table = mt.rate_table

In [59]:
rate_table = tables.add_aa_ia_dur_cols(rate_table)

 2025-06-06 00:25:01 | morai.experience.tables | INFO     | Removed '488' rows where attained_age, issue_age, or duration was invalid. 
Example: {'attained_age': 0, 'constant': 1, 'duration': 0, 'sex': 'F', 'smoker_status': 'NS', 'vals': 1.242311440660857e-06, 'issue_age': 1, 'vals_fixed': 1.242311440660857e-06, 'fixed_ind': 0} 


In [57]:
import importlib

importlib.reload(tables)

<module 'morai.experience.tables' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\tables.py'>

In [47]:
TableConstrainer = constraint.TableConstrainer(
    col_to_fix="vals",
    issue_age_col="issue_age",
    duration_col="duration",
    attained_age_col="attained_age",
    other_feature_cols=["sex", "smoker_status"],
    iteration_limit=10,
)

In [48]:
rate_table_fixed = TableConstrainer.fix_df(rate_table)

 2025-06-06 00:01:17 | morai.forecast.constraint | INFO     | Fixing dataframe on `vals` 
 2025-06-06 00:01:20 | morai.forecast.constraint | INFO     | iteration 1: 0 inner fixes + 0 outer fixes 
 2025-06-06 00:01:20 | morai.forecast.constraint | INFO     | COMPLETED with 0 iterations and total fixes: 0 


In [50]:
single_constraint = TableConstrainer.run_constraints(
    df=rate_table,
    constraints_list=[
        {
            "constraint_col": TableConstrainer.issue_age_col,
            "other_feature_cols": [
                TableConstrainer.duration_col,
                *TableConstrainer.other_feature_cols,
            ],
        }
    ],
)

 2025-06-06 00:02:05 | morai.forecast.constraint | INFO     | 0 out of 30008 rates need to be fixed for 'issue_age'. There are a total of 0 fixes. 


# Reload

In [14]:
importlib.reload(constraint)

<module 'morai.experience.tables' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\tables.py'>